# ZF / 美债期货期权链调试 notebook

这个 notebook 用 `ib_async` 逐步调试：连接 IB Gateway、枚举多个底层期货月份的 FOP 期权链、分批 qualify、并发订阅报价 / OI / 成交量 / Greeks。

关键点：不要只查 `ZF2606` 一个底层 conId。临近到期时，`ZF2606` 下面可能只剩少量期权到期日，后续日期通常挂在 `ZF2609`、`ZF2612` 等后续底层期货合约上。


In [2]:
from ib_async import IB, util
import pandas as pd
from pathlib import Path
import random

from treasury_fop_chain import (
    attach_error_collector,
    detach_error_collector,
    discover_future_months,
    qualify_future,
    get_future_price,
    get_future_prices_for_months,
    get_reference_price,
    build_treasury_fop_universe,
    save_universe,
    load_universe,
    FOPMarketDataStreamer,
    snapshot_in_batches,
    select_atm_contracts,
    filter_contracts_by_moneyness,
    prepare_snapshot_features,
    summarize_snapshot,
)

util.startLoop()
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)


## 1. 参数

`FUTURE_MONTHS` 是最重要的参数。比如当前要完整覆盖 ZF 期权，不要只填 `202606`，而是把后续活跃底层月份一起填上。

In [3]:
ROOT = "ZF"
FUTURE_MONTHS = ["202606", "202609", "202612"]

HOST = "127.0.0.1"
PORT = 4002          # IB Gateway paper: 4002; live: 4001; TWS paper: 7497
CLIENT_ID = random.randint(2000, 9999)
MARKET_DATA_TYPE = 1 # 1=live, 2=frozen, 3=delayed, 4=delayed frozen

UNIVERSE_CSV = Path(f"{ROOT}_FOP_Universe_{'_'.join(FUTURE_MONTHS)}.csv")
SNAPSHOT_CSV = Path(f"{ROOT}_FOP_Snapshot.csv")

ROOT, FUTURE_MONTHS, CLIENT_ID, UNIVERSE_CSV


('ZF',
 ['202606', '202609', '202612'],
 3902,
 WindowsPath('ZF_FOP_Universe_202606_202609_202612.csv'))

## 2. 连接 IB Gateway

如果看到 `10197`，通常是其他终端正在抢实时行情权限；如果看到 `354`，通常是没有对应市场数据订阅。

In [ ]:
ib = IB()
ib.connect(HOST, PORT, clientId=CLIENT_ID, timeout=10)
ib.reqMarketDataType(MARKET_DATA_TYPE)
errors, error_handler = attach_error_collector(ib)

print("connected:", ib.isConnected(), "clientId:", CLIENT_ID)


connected: True clientId: 3902


Error 10197, reqId 4: No market data during competing live session, contract: Future(conId=818615223, symbol='ZF', lastTradeDateOrContractMonth='20260630', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='ZFM6', tradingClass='ZF')


[IB error] 10197: No market data during competing live session


Error 200, reqId 1299: No security definition has been found for the request, contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.5, right='C', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Error 200, reqId 1300: No security definition has been found for the request, contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.5, right='P', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Error 200, reqId 1301: No security definition has been found for the request, contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.75, right='C', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')


[IB error] 200: No security definition has been found for the request
[IB error] 200: No security definition has been found for the request
[IB error] 200: No security definition has been found for the request


...

Error 10197, reqId 2576: No market data during competing live session, contract: Contract(secType='FOP', conId=849163266, symbol='ZF', lastTradeDateOrContractMonth='20260626', strike=105.75, right='C', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='OZFN6 C1057', tradingClass='OZF')
Error 10197, reqId 2726: No market data during competing live session, contract: Contract(secType='FOP', conId=842944961, symbol='ZF', lastTradeDateOrContractMonth='20260821', strike=103.25, right='P', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='OZFU6 P1032', tradingClass='OZF')
Error 10197, reqId 1930: No market data during competing live session, contract: Contract(secType='FOP', conId=882727427, symbol='ZF', lastTradeDateOrContractMonth='20260527', strike=104.25, right='C', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='WF4K6 C1042', tradingClass='WF4')
Error 10197, reqId 2159: No market data during competing live session, contract: Contract(secType

[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live 

Error 10197, reqId 2498: No market data during competing live session, contract: Contract(secType='FOP', conId=884728267, symbol='ZF', lastTradeDateOrContractMonth='20260608', strike=110.25, right='P', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='VF2M6 P1102', tradingClass='VF2')
Error 10197, reqId 2322: No market data during competing live session, contract: Contract(secType='FOP', conId=885751465, symbol='ZF', lastTradeDateOrContractMonth='20260604', strike=106.25, right='C', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='HF1M6 C1062', tradingClass='HF1')
Error 10197, reqId 1897: No market data during competing live session, contract: Contract(secType='FOP', conId=882370224, symbol='ZF', lastTradeDateOrContractMonth='20260526', strike=106.0, right='C', multiplier='1000', exchange='CBOT', currency='USD', localSymbol='GF4K6 C1060', tradingClass='GF4')
Error 10197, reqId 2497: No market data during competing live session, contract: Contract(secType=

[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live session
[IB error] 10197: No market data during competing live 

Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Peer closed connection.


## 3. Get current future price

Use this cell to fetch the current price of a specified Treasury futures contract.

In [6]:
price_info = get_future_price(
    ib,
    ROOT,
    FUTURE_MONTHS[0],
    market_data_type=MARKET_DATA_TYPE,
)

print("contract:", price_info["localSymbol"], "conId:", price_info["conId"])
print("price:", price_info["price"], "source:", price_info["priceSource"])
pd.Series({k: v for k, v in price_info.items() if k not in ["contract", "ticker"]})


contract: ZFM6 conId: 818615223
price: 107.0859375 source: marketPrice


root                    ZF
month               202606
exchange              CBOT
conId            818615223
localSymbol           ZFM6
price           107.085938
priceSource    marketPrice
marketPrice     107.085938
mid             107.082031
last            107.085938
close            106.78125
bid             107.078125
ask             107.085938
dtype: object

## 3. 可选：自动发现后续底层期货月份

如果你不确定应该填哪些月份，先跑这个单元看 IB 返回了哪些 FUT 合约月份。

In [7]:
# 可选：如果 FUTURE_MONTHS 不确定，打开下面两行。
discovered = discover_future_months(ib, ROOT, min_month="202606", max_count=10)
discovered


['202606', '202609', '202612']

## 4. 枚举并缓存全量期权链

这里会对每一个底层期货月份分别调用 `reqSecDefOptParams(root, 'CBOT', 'FUT', underlying.conId)`，再按返回的 `tradingClass / expirations / strikes` 构造 FOP 合约。分批 `qualifyContracts` 后会保存 conId，后续刷新行情可以直接从 CSV 加载，不必每次重新 qualify。

In [8]:
universe = build_treasury_fop_universe(
    ib,
    root=ROOT,
    future_months=FUTURE_MONTHS,
    exchange="CBOT",
    fop_exchange="CBOT",
    qualify_batch_size=400,
)

save_universe(universe, UNIVERSE_CSV)
print("valid FOP contracts:", len(universe.contracts))
print("saved:", UNIVERSE_CSV)
display(universe.chain_summary)
display(universe.metadata.head(20))


underlying ZF202606: conId=818615223, localSymbol=ZFM6
  candidates from option chains: 0
underlying ZF202609: conId=842590380, localSymbol=ZFU6
  candidates from option chains: 1592
underlying ZF202612: conId=869928209, localSymbol=ZFZ6
  candidates from option chains: 280


Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.5, right='C', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.5, right='P', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.75, right='C', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=88.75, right='P', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', strike=116.5, right='C', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260724', s

Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260925', strike=116.0, right='C', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')
Unknown contract: FuturesOption(symbol='ZF', lastTradeDateOrContractMonth='20260925', strike=116.0, right='P', multiplier='1000', exchange='CBOT', currency='USD', tradingClass='OZF')


qualified 1872/1872; valid 1852
valid FOP contracts: 1852
saved: ZF_FOP_Universe_202606_202609_202612.csv


,underlyingConId,underlyingLocalSymbol,underlyingMonth,exchange,tradingClass,multiplier,expirationCount,strikeCount,candidateCount,firstExpiration,lastExpiration,minStrike,maxStrike
0,842590380,ZFU6,202609,CBOT,VF1,1000,1,53,106,20260601,20260601,100.50,113.50
1,842590380,ZFU6,202609,CBOT,GF1,1000,1,50,100,20260602,20260602,100.50,112.75
2,842590380,ZFU6,202609,CBOT,WF1,1000,1,49,98,20260603,20260603,100.75,112.75
3,842590380,ZFU6,202609,CBOT,VF2,1000,1,50,100,20260608,20260608,100.50,112.75
4,842590380,ZFU6,202609,CBOT,HF1,1000,1,49,98,20260604,20260604,100.75,112.75
5,842590380,ZFU6,202609,CBOT,GF4,1000,1,52,104,20260526,20260526,100.50,113.25
6,842590380,ZFU6,202609,CBOT,WF4,1000,1,52,104,20260527,20260527,100.50,113.25
7,842590380,ZFU6,202609,CBOT,ZF1,1000,1,54,108,20260605,20260605,100.50,113.75
8,842590380,ZFU6,202609,CBOT,HF4,1000,1,52,104,20260528,20260528,100.50,113.25
9,842590380,ZFU6,202609,CBOT,ZF2,1000,1,50,100,20260612,20260612,100.50,112.75


,underlyingConId,underlyingLocalSymbol,underlyingMonth,chainExchange,chainTradingClass,chainMultiplier,conId,secType,symbol,localSymbol,tradingClass,lastTradeDateOrContractMonth,strike,right,multiplier,exchange,currency
0,842590380,ZFU6,202609,CBOT,GF4,1000,884726993,FOP,ZF,GF4K6 C1005,GF4,20260526,100.50,C,1000,CBOT,USD
1,842590380,ZFU6,202609,CBOT,GF4,1000,884600076,FOP,ZF,GF4K6 C1007,GF4,20260526,100.75,C,1000,CBOT,USD
2,842590380,ZFU6,202609,CBOT,GF4,1000,884600082,FOP,ZF,GF4K6 C1010,GF4,20260526,101.00,C,1000,CBOT,USD
3,842590380,ZFU6,202609,CBOT,GF4,1000,882369874,FOP,ZF,GF4K6 C1012,GF4,20260526,101.25,C,1000,CBOT,USD
4,842590380,ZFU6,202609,CBOT,GF4,1000,882369988,FOP,ZF,GF4K6 C1015,GF4,20260526,101.50,C,1000,CBOT,USD
5,842590380,ZFU6,202609,CBOT,GF4,1000,882369892,FOP,ZF,GF4K6 C1017,GF4,20260526,101.75,C,1000,CBOT,USD
6,842590380,ZFU6,202609,CBOT,GF4,1000,882369885,FOP,ZF,GF4K6 C1020,GF4,20260526,102.00,C,1000,CBOT,USD
7,842590380,ZFU6,202609,CBOT,GF4,1000,882370185,FOP,ZF,GF4K6 C1022,GF4,20260526,102.25,C,1000,CBOT,USD
8,842590380,ZFU6,202609,CBOT,GF4,1000,882370234,FOP,ZF,GF4K6 C1025,GF4,20260526,102.50,C,1000,CBOT,USD
9,842590380,ZFU6,202609,CBOT,GF4,1000,882370079,FOP,ZF,GF4K6 C1027,GF4,20260526,102.75,C,1000,CBOT,USD


## 5. 从缓存加载 universe

调试 market data 时建议从这里开始反复跑，减少 IB contract details 请求。

In [9]:
contracts, universe_df = load_universe(UNIVERSE_CSV)
universe_df["expiration"] = universe_df["lastTradeDateOrContractMonth"].astype(str)

summary = universe_df.groupby("underlyingMonth", dropna=False).agg(
    contracts=("conId", "count"),
    expirations=("expiration", "nunique"),
    firstExpiration=("expiration", "min"),
    lastExpiration=("expiration", "max"),
    minStrike=("strike", "min"),
    maxStrike=("strike", "max"),
)

print("loaded contracts:", len(contracts))
display(summary)
display(universe_df)


loaded contracts: 1852


,contracts,expirations,firstExpiration,lastExpiration,minStrike,maxStrike
underlyingMonth,,,,,,
202609,1574,14,20260526,20260821,88.50,117.5
202612,278,2,20260925,20261120,98.75,116.0


,underlyingConId,underlyingLocalSymbol,underlyingMonth,chainExchange,chainTradingClass,chainMultiplier,conId,secType,symbol,localSymbol,tradingClass,lastTradeDateOrContractMonth,strike,right,multiplier,exchange,currency,expiration
0,842590380,ZFU6,202609,CBOT,GF4,1000,884726993,FOP,ZF,GF4K6 C1005,GF4,20260526,100.50,C,1000,CBOT,USD,20260526
1,842590380,ZFU6,202609,CBOT,GF4,1000,884600076,FOP,ZF,GF4K6 C1007,GF4,20260526,100.75,C,1000,CBOT,USD,20260526
2,842590380,ZFU6,202609,CBOT,GF4,1000,884600082,FOP,ZF,GF4K6 C1010,GF4,20260526,101.00,C,1000,CBOT,USD,20260526
3,842590380,ZFU6,202609,CBOT,GF4,1000,882369874,FOP,ZF,GF4K6 C1012,GF4,20260526,101.25,C,1000,CBOT,USD,20260526
4,842590380,ZFU6,202609,CBOT,GF4,1000,882369988,FOP,ZF,GF4K6 C1015,GF4,20260526,101.50,C,1000,CBOT,USD,20260526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1847,869928209,ZFZ6,202612,CBOT,OZF,1000,870222700,FOP,ZF,OZFZ6 P1150,OZF,20261120,115.00,P,1000,CBOT,USD,20261120
1848,869928209,ZFZ6,202612,CBOT,OZF,1000,870222757,FOP,ZF,OZFZ6 P1152,OZF,20261120,115.25,P,1000,CBOT,USD,20261120
1849,869928209,ZFZ6,202612,CBOT,OZF,1000,870222895,FOP,ZF,OZFZ6 P1155,OZF,20261120,115.50,P,1000,CBOT,USD,20261120
1850,869928209,ZFZ6,202612,CBOT,OZF,1000,870223010,FOP,ZF,OZFZ6 P1157,OZF,20261120,115.75,P,1000,CBOT,USD,20261120


## 6. 过滤需要监控的合约

规则：0DTE 合约只保留距离当前期货价格 2 点以内的 strike；非 0DTE 合约保留 5 点以内。不同底层月份会分别使用对应期货合约的价格。

In [10]:
future_prices = get_future_prices_for_months(
    ib,
    ROOT,
    FUTURE_MONTHS,
    market_data_type=MARKET_DATA_TYPE,
)
spot_by_month = dict(zip(future_prices["month"].astype(str), future_prices["price"].astype(float)))

filtered_contracts, filtered_universe_df = filter_contracts_by_moneyness(
    contracts,
    universe_df,
    spot_by_underlying_month=spot_by_month,
    dte0_width=2.0,
    non_dte0_width=4.0,
)

print("original contracts:", len(contracts))
print("filtered contracts:", len(filtered_contracts))
display(future_prices)
display(
    filtered_universe_df.groupby(["underlyingMonth", "expiration", "dte0"], dropna=False)
    .agg(contracts=("conId", "count"), minStrike=("strike", "min"), maxStrike=("strike", "max"))
    .head(30)
)


original contracts: 1852
filtered contracts: 992


,root,month,exchange,conId,localSymbol,price,priceSource,marketPrice,mid,last,close,bid,ask
0,ZF,202606,CBOT,818615223,ZFM6,107.085938,marketPrice,107.085938,107.082031,107.085938,106.781250,107.078125,107.085938
1,ZF,202609,CBOT,842590380,ZFU6,106.984375,marketPrice,106.984375,106.980469,106.984375,106.664062,106.976562,106.984375
2,ZF,202612,CBOT,869928209,ZFZ6,106.976562,marketPrice,106.976562,106.894531,106.976562,106.640625,106.796875,106.992188


contracts  minStrike  maxStrike
underlyingMonth expiration dte0                                  
202609          20260526   True          32      105.0     108.75
                20260527   False         64      103.0     110.75
                20260528   False         64      103.0     110.75
                20260529   False         64      103.0     110.75
                20260601   False         64      103.0     110.75
                20260602   False         64      103.0     110.75
                20260603   False         64      103.0     110.75
                20260604   False         64      103.0     110.75
                20260605   False         64      103.0     110.75
                20260608   False         64      103.0     110.75
                20260612   False         64      103.0     110.75
                20260626   False         64      103.0     110.75
                20260724   False         64      103.0     110.75
                20260821   False         64      103.0     110.75
202612          20260925   False         64      103.0     110.75
                20261120   False         64      103.0     110.75

## 6. 小样本测试报价 / Greeks

先用 80 个合约确认权限、字段和到达速度。`genericTickList='100,101,106'` 会请求期权成交量、OI、IV/Greeks。

In [ ]:
# 不要用 contracts[:80] 做样本：排序最前面通常全是当日到期合约，IB 很容易返回 Model is not valid。
# 这里改为：优先从过滤后的 universe 中选非当日到期、ATM 附近、最近几个到期日。
today = pd.Timestamp.now(tz="Asia/Shanghai").strftime("%Y%m%d")
front_future = qualify_future(ib, ROOT, FUTURE_MONTHS[0])
fallback_spot = float(pd.to_numeric(universe_df["strike"], errors="coerce").median())
spot = get_reference_price(ib, front_future, fallback=fallback_spot)

sample_source_contracts = filtered_contracts if "filtered_contracts" in globals() else contracts
sample_source_df = filtered_universe_df if "filtered_universe_df" in globals() else universe_df

sample_contracts, sample_meta = select_atm_contracts(
    sample_source_contracts,
    sample_source_df,
    spot=spot,
    expiration_after=today,
    underlying_month=FUTURE_MONTHS[0],
    max_expirations=3,
    strikes_each_side=5,
)

print("today:", today, "spot:", spot, "sample size:", len(sample_contracts))
display(sample_meta[["underlyingMonth", "localSymbol", "expiration", "right", "strike", "conId"]].head(30))

sample_streamer = FOPMarketDataStreamer(ib, request_interval=0.025)
sample_streamer.subscribe(sample_contracts)
sample_stats = sample_streamer.wait_until_stable(min_seconds=5, max_seconds=20, stable_seconds=3)
sample_df = sample_streamer.snapshot()
sample_streamer.cancel()
sample_features = prepare_snapshot_features(sample_df, include_chinese_columns=True)

display(sample_stats)
display(summarize_snapshot(sample_df))
display(sample_features.head(20))

## 7. 全量并发订阅：最快刷新方式

这一步会保持所有 market data line 打开。之后每次 `streamer.snapshot()` 都只是读取内存中已经被 IB event loop 更新的 Ticker，速度最快。

如果这里报 market data line 上限，改用下一节的分批模式，或把 `active_contracts = contracts[:300]` 改小。

In [11]:
active_contracts = filtered_contracts if "filtered_contracts" in globals() else contracts
print("subscribing contracts:", len(active_contracts))

streamer = FOPMarketDataStreamer(ib, request_interval=0.01)
streamer.subscribe(active_contracts)
stats = streamer.wait_until_stable(max_seconds=20, stable_seconds=2)

df = streamer.snapshot()
df_features = prepare_snapshot_features(df, include_chinese_columns=True)
df.to_csv(SNAPSHOT_CSV, index=False, encoding="utf-8-sig")
FEATURES_CSV = SNAPSHOT_CSV.with_name(SNAPSHOT_CSV.stem + "_features_cn.csv")
df_features.to_csv(FEATURES_CSV, index=False, encoding="utf-8-sig")

display(stats)
display(summarize_snapshot(df))
display(df_features.head(20))
print("raw saved:", SNAPSHOT_CSV)
print("features saved:", FEATURES_CSV)

subscribing contracts: 992
sent 992/992 market data requests in 7.6s
ready quote=992/992, greeks=957/992, oi=893/992, volume=0/992, elapsed=18.9s


StreamStats(requested=992, quote_ready=992, greek_ready=957, oi_ready=893, volume_ready=0, elapsed_seconds=18.875)

contracts                  992
with_quote                 992
with_model_greeks          957
with_primary_delta         957
with_open_interest         893
with_volume                992
expirations                 16
min_expiration        20260526
max_expiration        20261120
dtype: object

,快照时间_北京时间,距离到期天数,快照时间_UTC,到期日,行权价,看涨看跌,买价,卖价,买卖中间价,最新成交价,昨收价,买量,卖量,未平仓量,IB期权隐含波动率tick,模型隐含波动率,Delta,Gamma,Theta,Vega,模型期权价,模型标的价
0,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,105.00,C,1.968750,1.984375,1.976562,NaN,1.664062,1.0,1.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,105.25,C,1.718750,1.734375,1.726562,NaN,1.414062,1.0,1.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,105.50,C,1.468750,1.484375,1.476562,NaN,1.164062,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,105.75,C,1.218750,1.234375,1.226562,NaN,0.914062,1.0,1.0,NaN,0.084539,0.105256,9.985622e-01,1.144594e-02,-1.551028e-04,2.918950e-04,1.226718e+00,106.976562
4,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,106.00,C,0.968750,0.984375,0.976562,NaN,0.671875,26.0,36.0,NaN,0.066520,0.093895,9.960245e-01,3.175578e-02,-4.436633e-04,6.785561e-04,9.770062e-01,106.976562
5,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,106.25,C,0.718750,0.734375,0.726562,NaN,0.437500,36.0,36.0,417.0,0.050530,0.080869,9.887261e-01,9.211757e-02,-1.235987e-03,1.555297e-03,7.277985e-01,106.976562
6,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,106.50,C,0.468750,0.484375,0.476562,0.570312,0.242188,61.0,1.0,904.0,0.033078,0.066814,9.651419e-01,2.916339e-01,-3.655082e-03,3.939905e-03,4.802176e-01,106.976562
7,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,106.75,C,0.234375,0.250000,0.242188,NaN,0.109375,26.0,1.0,5448.0,0.053483,0.055441,8.493612e-01,1.066557e+00,-1.723692e-02,1.025360e-02,2.437994e-01,106.976562
8,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,107.00,C,0.062500,0.078125,0.070312,0.078125,0.039062,33989.0,15359.0,20231.0,0.051225,0.051905,4.551209e-01,1.928165e+00,-7.124346e-02,1.580375e-02,7.124346e-02,106.976562
9,2026-05-26 14:56:11,0,2026-05-26 06:56:11.348822+0000,20260526,107.25,C,0.007812,0.015625,0.011719,NaN,0.007812,23226.0,160289.0,3361.0,0.055888,0.056933,1.126527e-01,8.508107e-01,-1.236316e-02,8.603107e-03,1.236316e-02,106.976562


raw saved: ZF_FOP_Snapshot.csv
features saved: ZF_FOP_Snapshot_features_cn.csv


## 8. 快速刷新快照

保持上一节 `streamer` 不 cancel，然后反复运行这个单元即可刷新整条链的最新报价和 Greeks。

In [13]:
# ib.sleep(2)
df_refresh = streamer.snapshot()
df_refresh_features = prepare_snapshot_features(df_refresh, include_chinese_columns=True)
df_refresh.to_csv(SNAPSHOT_CSV, index=False, encoding="utf-8-sig")
FEATURES_CSV = SNAPSHOT_CSV.with_name(SNAPSHOT_CSV.stem + "_features_cn.csv")
df_refresh_features.to_csv(FEATURES_CSV, index=False, encoding="utf-8-sig")

display(summarize_snapshot(df_refresh))
display(df_refresh_features.head(20))
print("features saved:", FEATURES_CSV)

contracts                  992
with_quote                 992
with_model_greeks          992
with_primary_delta         992
with_open_interest         893
with_volume                992
expirations                 16
min_expiration        20260526
max_expiration        20261120
dtype: object

,快照时间_北京时间,距离到期天数,快照时间_UTC,到期日,行权价,看涨看跌,买价,卖价,买卖中间价,最新成交价,昨收价,买量,卖量,未平仓量,IB期权隐含波动率tick,模型隐含波动率,Delta,Gamma,Theta,Vega,模型期权价,模型标的价
0,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,105.00,C,1.968750,1.984375,1.976562,NaN,1.664062,1.0,1.0,9.0,NaN,0.141192,9.998961e-01,1.170873e-03,-6.232187e-06,3.277581e-05,1.976569e+00,106.976562
1,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,105.25,C,1.718750,1.734375,1.726562,NaN,1.414062,1.0,1.0,16.0,NaN,0.128177,9.997832e-01,2.011759e-03,-1.684266e-05,5.832362e-05,1.726579e+00,106.976562
2,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,105.50,C,1.468750,1.484375,1.476562,NaN,1.164062,1.0,1.0,NaN,NaN,0.116086,9.994898e-01,4.269791e-03,-4.818338e-05,1.199880e-04,1.476611e+00,106.976562
3,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,105.75,C,1.218750,1.234375,1.226562,NaN,0.914062,1.0,1.0,NaN,0.084539,0.104849,9.986384e-01,1.096570e-02,-1.447391e-04,2.831922e-04,1.226707e+00,106.976562
4,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,106.00,C,0.968750,0.984375,0.976562,NaN,0.671875,1.0,36.0,NaN,0.066520,0.093577,9.961320e-01,3.109827e-02,-4.286247e-04,6.509451e-04,9.769911e-01,106.976562
5,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,106.25,C,0.718750,0.734375,0.726562,NaN,0.437500,26.0,36.0,417.0,0.050530,0.080739,9.888712e-01,9.127895e-02,-1.215567e-03,1.542532e-03,7.277781e-01,106.976562
6,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,106.50,C,0.468750,0.484375,0.476562,0.570312,0.242188,61.0,36.0,904.0,0.033078,0.066988,9.648271e-01,2.931221e-01,-3.701551e-03,3.955032e-03,4.802641e-01,106.976562
7,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,106.75,C,0.234375,0.250000,0.242188,NaN,0.109375,26.0,1.0,5448.0,0.053483,0.055767,8.482073e-01,1.066868e+00,-1.748504e-02,1.030131e-02,2.440475e-01,106.976562
8,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,107.00,C,0.062500,0.078125,0.070312,0.078125,0.039062,22930.0,15413.0,20231.0,0.051225,0.052079,4.552418e-01,1.923118e+00,-7.146125e-02,1.579266e-02,7.146125e-02,106.976562
9,2026-05-26 14:57:09,0,2026-05-26 06:57:09.344818+0000,20260526,107.25,C,0.007812,0.015625,0.011719,NaN,0.007812,23233.0,160539.0,3361.0,0.055888,0.056914,1.124402e-01,8.504477e-01,-1.232346e-02,8.586190e-03,1.232346e-02,106.976562


features saved: ZF_FOP_Snapshot_features_cn.csv


## 高频期货价格刷新

In [ ]:
from IPython.display import clear_output, display
import pandas as pd
import time

FUTURE_REFRESH_SECONDS = 0.5
FUTURE_STREAM_SECONDS = 120

if "future_tickers" in globals():
    for old_ticker in future_tickers:
        try:
            ib.cancelMktData(old_ticker.contract)
        except Exception:
            pass
    ib.sleep(0.5)

future_tickers = []
for month in FUTURE_MONTHS:
    fut = qualify_future(ib, ROOT, month, exchange="CBOT")
    ticker = ib.reqMktData(fut, genericTickList="", snapshot=False)
    ticker._month = str(month)
    future_tickers.append(ticker)

rows_history = []
start = time.time()

while time.time() - start < FUTURE_STREAM_SECONDS:
    ib.sleep(FUTURE_REFRESH_SECONDS)

    rows = []
    ts = pd.Timestamp.now(tz="Asia/Shanghai")
    for ticker in future_tickers:
        bid = ticker.bid
        ask = ticker.ask
        last = ticker.last
        close = ticker.close
        market_price = ticker.marketPrice()

        rows.append({
            "timeChina": ts.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
            "month": ticker._month,
            "localSymbol": ticker.contract.localSymbol,
            "bid": bid,
            "ask": ask,
            "last": last,
            "close": close,
            "marketPrice": market_price,
            "bidSize": ticker.bidSize,
            "askSize": ticker.askSize,
            "lastSize": ticker.lastSize,
        })

    future_live_df = pd.DataFrame(rows)
    rows_history.extend(rows)

    clear_output(wait=True)
    display(future_live_df)

future_price_ticks = pd.DataFrame(rows_history)
FUTURE_TICKS_CSV = Path(f"{ROOT}_Future_Live_Ticks.csv")
future_price_ticks.to_csv(FUTURE_TICKS_CSV, index=False, encoding="utf-8-sig")
print("saved:", FUTURE_TICKS_CSV)

## 9. 备选：分批并发快照

如果账户行情线数量不足，无法全量保持订阅，就用这个模式。它比保持订阅慢，但每批内部仍然是并发 market data streams。

In [ ]:
# 如需使用分批模式，先取消上一节的 streamer。
# streamer.cancel()
#
# batch_contracts = filtered_contracts if "filtered_contracts" in globals() else contracts
# df_batch = snapshot_in_batches(
#     ib,
#     batch_contracts,
#     batch_size=300,
#     wait_max_seconds=15,
#     wait_stable_seconds=2,
# )
# df_batch_features = prepare_snapshot_features(df_batch, include_chinese_columns=True)
# df_batch.to_csv(SNAPSHOT_CSV, index=False, encoding="utf-8-sig")
# FEATURES_CSV = SNAPSHOT_CSV.with_name(SNAPSHOT_CSV.stem + "_features_cn.csv")
# df_batch_features.to_csv(FEATURES_CSV, index=False, encoding="utf-8-sig")
# display(summarize_snapshot(df_batch))
# display(df_batch_features.head(20))

## 10. 查看 IB API 错误

重点看 `10197`、`354`、`420`、`100/101`。这些通常分别对应其他终端抢权限、无行情权限、节流/限速、消息速率或行情线限制。

In [ ]:
pd.DataFrame(errors).tail(30)


## 11. 清理连接

跑完一定取消 market data，不然 IB Gateway 会继续占用行情线。

In [ ]:
for name in ["streamer", "sample_streamer"]:
    obj = globals().get(name)
    if obj is not None:
        obj.cancel()

if "error_handler" in globals():
    detach_error_collector(ib, error_handler)

if ib.isConnected():
    ib.disconnect()

print("disconnected")


## 12. 可视化测试（无需 Streamlit）

把 `zf_viz.py` 里的图表函数直接在 Jupyter 里渲染。
- HTML 期权链用 `display(HTML(...))`
- Altair 图直接 `.display()` 或把变量放在 cell 末尾

**前提**：已跑过第 7/8 节拿到 `df_features`，或直接从 CSV 加载。

In [14]:
from IPython.display import display, HTML
import pandas as pd
from zf_viz import (
    build_chain_html,
    chart_iv_smile,
    chart_heatmap,
    chart_oi_bars,
    chart_flow_heatmap,
    chart_flow_by_strike,
)

# ── 数据源：优先用内存里已有的 df_features；否则从 CSV 加载 ──
if "df_features" in globals() and not df_features.empty:
    snap = df_features.copy()
    print("使用内存 df_features，行数:", len(snap))
elif "df_refresh_features" in globals() and not df_refresh_features.empty:
    snap = df_refresh_features.copy()
    print("使用内存 df_refresh_features，行数:", len(snap))
else:
    snap = pd.read_csv(FEATURES_CSV)
    print("从 CSV 加载:", FEATURES_CSV, "，行数:", len(snap))

# 统一列名到 dashboard 期望的英文列
col_map = {
    "到期日": "expiration", "行权价": "strike", "看涨看跌": "right",
    "买价": "bid", "卖价": "ask", "买卖中间价": "mid", "最新成交价": "last",
    "未平仓量": "openInterest", "模型隐含波动率": "iv",
    "Delta": "delta", "Gamma": "gamma", "Theta": "theta", "Vega": "vega",
}
for cn, en in col_map.items():
    if cn in snap.columns and en not in snap.columns:
        snap[en] = snap[cn]

for col in ["strike", "bid", "ask", "mid", "openInterest", "iv", "delta"]:
    if col in snap.columns:
        snap[col] = pd.to_numeric(snap[col], errors="coerce")
snap["expiration"] = snap["expiration"].astype(str)
snap["right"]      = snap["right"].astype(str)

expirations = sorted(snap["expiration"].dropna().unique())
print("可用到期日：", expirations[:10], "... 共", len(expirations), "个")

# ── ATM 参考价（用模型标的价中位数，或手工填）──
if "undPrice" in snap.columns:
    spot = float(pd.to_numeric(snap["undPrice"], errors="coerce").dropna().median())
elif "模型标的价" in snap.columns:
    spot = float(pd.to_numeric(snap["模型标的价"], errors="coerce").dropna().median())
else:
    spot = None
print("ATM spot:", spot)

使用内存 df_features，行数: 992
可用到期日： ['20260526', '20260527', '20260528', '20260529', '20260601', '20260602', '20260603', '20260604', '20260605', '20260608'] ... 共 16 个
ATM spot: 106.9765625


In [18]:
# ── 期权链 HTML 表格 ──────────────────────────────────────────────────────────
# 改 selected_exp 查不同到期日
selected_exp = expirations[0]
print("显示到期日：", selected_exp, "  spot:", spot)
display(HTML(build_chain_html(snap, selected_exp, spot)))

显示到期日： 20260526   spot: 106.9765625


IV,Delta,OI,Bid,Ask,Mid,Strike,Mid,Ask,Bid,OI,Delta,IV
,,9,1.9688,1.9844,1.9766,105.00,,0.0078,,68,-0.000,0.142
,,16,1.7188,1.7344,1.7266,105.25,,0.0078,,69,-0.000,0.129
,,,1.4688,1.4844,1.4766,105.50,,0.0078,,70,-0.001,0.117
0.105,0.999,,1.2188,1.2344,1.2266,105.75,,0.0078,,916,-0.001,0.105
0.094,0.996,,0.9688,0.9844,0.9766,106.00,,0.0078,,"6,259",-0.004,0.094
0.081,0.989,417,0.7188,0.7344,0.7266,106.25,,0.0078,,"3,990",-0.011,0.081
0.067,0.965,904,0.4688,0.4844,0.4766,106.50,,0.0078,,"1,798",-0.035,0.067
0.055,0.849,"5,448",0.2344,0.2500,0.2422,106.75,0.0156,0.0234,0.0078,"3,979",-0.151,0.055
0.052,0.455,"20,231",0.0625,0.0781,0.0703,107.00,0.0938,0.1016,0.0859,"15,000",-0.545,0.052
0.057,0.113,"3,361",0.0078,0.0156,0.0117,107.25,0.2852,0.2969,0.2734,505,-0.887,0.057


In [19]:
# ── IV Smile + OI 柱状图 ──────────────────────────────────────────────────────
iv_chart = chart_iv_smile(snap, selected_exp)
if iv_chart:
    iv_chart.display()

oi_chart = chart_oi_bars(snap, selected_exp)
if oi_chart:
    oi_chart.display()

alt.Chart(...)

alt.Chart(...)

In [20]:
# ── 跨到期日热力图（IV / OI）────────────────────────────────────────────────
# 只看一个方向，避免图太大；改 right_filter = "P" 看看跌
right_filter = "C"
sub = snap[snap["right"] == right_filter]

iv_heatmap = chart_heatmap(sub, "iv", "IV")
if iv_heatmap:
    iv_heatmap.display()

oi_heatmap = chart_heatmap(sub, "openInterest", "OI")
if oi_heatmap:
    oi_heatmap.display()

alt.FacetChart(...)

alt.FacetChart(...)

In [ ]:
# ── Flow 散点 / 按 Strike 柱状图（需要先跑 SQLite flow 保存）───────────────
from treasury_fop_chain import load_flow_events_sqlite
from pathlib import Path

FLOW_DB = Path("data") / "zf_option_flow.sqlite"
flow = load_flow_events_sqlite(FLOW_DB, limit=5000) if FLOW_DB.exists() else pd.DataFrame()
print("flow events:", len(flow))

if not flow.empty:
    fh = chart_flow_heatmap(flow)
    if fh:
        fh.display()

    fb = chart_flow_by_strike(flow)
    if fb:
        fb.display()
else:
    print("暂无 flow 数据，先跑第 7/8 节让 Streamlit 或 refresh cell 写入 SQLite。")